# EmoFlow reproducibility notebook
# Author: Hiram Calvo

Run from top to bottom after installing `requirements.txt` and placing the required files in `data/`. The two experimental sections repeat some helper functions deliberately so that the frozen and fine-tuned regimes remain independently inspectable.

## Frozen-backbone experiments

In [ ]:
# ============================================================
# Frozen-backbone StoryCommonsense EmoFlow experiments
# BERT-tiny and DistilBERT; five manuscript strategies; four tasks
# ============================================================


import os, re, ast, json, hashlib, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from joblib import dump

warnings.filterwarnings("ignore")

PROJECT_DIR = os.path.abspath(os.environ.get("EMOFLOW_PROJECT_DIR", "."))
DATA_DIR = os.path.abspath(os.environ.get("EMOFLOW_DATA_DIR", os.path.join(PROJECT_DIR, "data")))
TRAIN_PATH = os.path.join(DATA_DIR, "storycs_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "storycs_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "storycs_test.csv")

CACHE_DIR = os.path.join(PROJECT_DIR, "cache", "frozen")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results", "frozen")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

VAD_LEXICON_PATH = os.path.abspath(
    os.environ.get(
        "NRC_VAD_PATH",
        os.path.join(DATA_DIR, "unigrams-NRC-VAD-Lexicon-v2.1.txt")
    )
)

required_files = [TRAIN_PATH, DEV_PATH, TEST_PATH, VAD_LEXICON_PATH]
missing_files = [path for path in required_files if not os.path.exists(path)]
if missing_files:
    raise FileNotFoundError(
        "Missing required input files. See README.md:\n" + "\n".join(missing_files)
    )

PLUTCHIK = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]

BACKBONES = {
    "bert_tiny": "prajjwal1/bert-tiny",
    "distilbert": "distilbert-base-uncased"

}

TASKS = [
    "state_current",
    "forecast_next",
    "change_subset",
    "forecast_change_subset",
]

VADBERT_MODEL_NAME = "RobroKools/vad-bert"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

train_raw = pd.read_csv(TRAIN_PATH)
dev_raw   = pd.read_csv(DEV_PATH)
test_raw  = pd.read_csv(TEST_PATH)

print("train:", train_raw.shape)
print("dev:", dev_raw.shape)
print("test:", test_raw.shape)

def safe_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        y = ast.literal_eval(str(x))
        return y if isinstance(y, list) else [str(y)]
    except Exception:
        return [str(x)]

def parse_plutchik(x):
    vals = np.zeros(len(PLUTCHIK), dtype=np.float32)
    for item in safe_list(x):
        s = str(item).strip().lower()
        if s == "none":
            continue
        if ":" in s:
            lab, score = s.split(":", 1)
            lab = lab.strip()
            try:
                score = float(score)
            except Exception:
                score = 1.0
        else:
            lab, score = s.strip(), 3.0
        if lab in PLUTCHIK:
            vals[PLUTCHIK.index(lab)] = max(vals[PLUTCHIK.index(lab)], score / 3.0)
    return vals

def binarize_y(y, threshold=1e-6):
    return (y > threshold).astype(int)

def clean_text(x):
    return "" if pd.isna(x) else str(x)

def make_full_text(row):
    ctx = clean_text(row.get("context", ""))
    sent = clean_text(row.get("sentence", ""))
    return (ctx.strip() + " " + sent.strip()).strip() if ctx.strip() else sent.strip()

def make_prev_text(row):
    return clean_text(row.get("context", "")).strip()

def row_hash(texts, prefix):
    h = hashlib.md5()
    for t in texts:
        h.update(str(t).encode("utf-8", errors="ignore"))
    return prefix + "_" + h.hexdigest()[:16]

def cache_npy_path(name):
    safe = re.sub(r"[^a-zA-Z0-9_\-\.]+", "_", name)
    return os.path.join(CACHE_DIR, safe + ".npy")

def aggregate_annotations(df):
    df = df.copy()
    df["full_text"] = df.apply(make_full_text, axis=1)
    df["prev_text"] = df.apply(make_prev_text, axis=1)
    df["plutchik_vec"] = df["plutchik"].apply(parse_plutchik)

    rows = []
    group_cols = ["storyid", "linenum", "char"]

    for key, g in df.groupby(group_cols, sort=False):
        y = np.vstack(g["plutchik_vec"].values).mean(axis=0)
        first = g.iloc[0]
        rows.append({
            "storyid": key[0],
            "linenum": key[1],
            "char": key[2],
            "sentence": clean_text(first["sentence"]),
            "context": clean_text(first["context"]),
            "full_text": clean_text(first["full_text"]),
            "prev_text": clean_text(first["prev_text"]),
            "affected_yes_ratio": np.mean(g["affected"].astype(str).str.lower().eq("yes")),
            "y": y,
        })

    out = pd.DataFrame(rows)
    return out.sort_values(["storyid", "char", "linenum"]).reset_index(drop=True)

def add_previous_targets(df):
    df = df.copy()
    prevs = []

    for _, g in df.groupby(["storyid", "char"], sort=False):
        ys = list(g["y"].values)
        idxs = list(g.index)
        for pos, idx in enumerate(idxs):
            if pos == 0:
                prevs.append((idx, np.zeros(len(PLUTCHIK), dtype=np.float32), False))
            else:
                prevs.append((idx, ys[pos-1], True))

    prev_map = {idx: (vec, has_prev) for idx, vec, has_prev in prevs}
    df["y_prev"] = df.index.map(lambda i: prev_map[i][0])
    df["has_prev"] = df.index.map(lambda i: prev_map[i][1])
    df["changed_from_prev"] = df.apply(
        lambda r: bool(np.any(binarize_y(r["y"]) != binarize_y(r["y_prev"]))) if r["has_prev"] else False,
        axis=1
    )
    return df

dev = add_previous_targets(aggregate_annotations(dev_raw))
test = add_previous_targets(aggregate_annotations(test_raw))

print("Aggregated dev:", dev.shape)
print("Aggregated test:", test.shape)

def build_task(df, task_name):
    df = df.copy()

    if task_name == "state_current":
        df["input_text"] = df.apply(
            lambda r: f"Character: {r['char']}. Story: {r['full_text']}",
            axis=1
        )
        use = df.copy()

    elif task_name == "forecast_next":
        use = df[df["has_prev"] & df["prev_text"].astype(str).str.len().gt(0)].copy()
        use["input_text"] = use.apply(
            lambda r: f"Character: {r['char']}. Previous story: {r['prev_text']}",
            axis=1
        )

    elif task_name == "change_subset":
        use = df[df["has_prev"] & df["changed_from_prev"]].copy()
        use["input_text"] = use.apply(
            lambda r: f"Character: {r['char']}. Story: {r['full_text']}",
            axis=1
        )

    elif task_name == "forecast_change_subset":
        use = df[
            df["has_prev"] &
            df["changed_from_prev"] &
            df["prev_text"].astype(str).str.len().gt(0)
        ].copy()
        use["input_text"] = use.apply(
            lambda r: f"Character: {r['char']}. Previous story: {r['prev_text']}",
            axis=1
        )

    else:
        raise ValueError(f"Unknown task: {task_name}")

    use = use.reset_index(drop=True)
    texts = use["input_text"].tolist()
    y = binarize_y(np.vstack(use["y"].values))

    return use, texts, y

def get_transformer_cls_features(texts, model_name, cache_name, batch_size=32, max_length=192):
    cache_path = cache_npy_path(cache_name)
    if os.path.exists(cache_path):
        return np.load(cache_path)

    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE)
    model.eval()

    feats = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"CLS {cache_name}"):
            batch = texts[i:i+batch_size]
            enc = tok(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(DEVICE)
            out = model(**enc).last_hidden_state[:, 0, :]
            feats.append(out.detach().cpu().numpy())

    feats = np.vstack(feats).astype(np.float32)
    np.save(cache_path, feats)
    return feats

def split_context_sentences(context, current_sentence=None):
    parts = []
    ctx = clean_text(context)

    if ctx.strip():
        rough = re.split(r"(?<=[\.\!\?])\s+", ctx)
        if len(rough) <= 1:
            rough = ctx.replace("!", ".").replace("?", ".").split(".")
        parts += [p.strip() for p in rough if p.strip()]

    if current_sentence is not None and clean_text(current_sentence).strip():
        parts.append(clean_text(current_sentence).strip())

    return parts

def load_nrc_vad(path):
    lex = {}
    if not os.path.exists(path):
        print("NRC VAD lexicon not found. VAD-lex features will be zeros:", path)
        return lex

    tmp = pd.read_csv(path, sep="\t")
    tmp.columns = [c.lower() for c in tmp.columns]

    word_col = "word" if "word" in tmp.columns else tmp.columns[0]
    v_col = "valence" if "valence" in tmp.columns else tmp.columns[1]
    a_col = "arousal" if "arousal" in tmp.columns else tmp.columns[2]
    d_col = "dominance" if "dominance" in tmp.columns else tmp.columns[3]

    for _, r in tmp.iterrows():
        lex[str(r[word_col]).lower()] = np.array(
            [float(r[v_col]), float(r[a_col]), float(r[d_col])],
            dtype=np.float32
        )

    print("Loaded NRC VAD entries:", len(lex))
    return lex

VAD_LEX = load_nrc_vad(VAD_LEXICON_PATH)

def simple_tokenize(text):
    return [w.strip(".,;:!?()[]{}\"'").lower() for w in str(text).split()]

def vad_lex_text(text):
    vals = [VAD_LEX[w] for w in simple_tokenize(text) if w in VAD_LEX]
    if not vals:
        return np.zeros(3, dtype=np.float32)
    return np.vstack(vals).mean(axis=0).astype(np.float32)

def sequence_to_flow(seq, dim):
    if len(seq) == 0:
        seq = [np.zeros(dim, dtype=np.float32)]

    seq = [np.asarray(x, dtype=np.float32) for x in seq]

    while len(seq) < 3:
        seq.insert(0, np.zeros(dim, dtype=np.float32))

    states = seq[-3:]
    deltas = [states[1] - states[0], states[2] - states[1]]
    global_state = np.vstack(seq).mean(axis=0)

    return np.concatenate(states + deltas + [global_state], axis=0).astype(np.float32)

def emoflow_vadlex_features(df, cache_name, include_current=True):
    cache_path = cache_npy_path(cache_name)
    if os.path.exists(cache_path):
        return np.load(cache_path)

    feats = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"VAD-lex flow {cache_name}"):
        parts = split_context_sentences(r["context"], r["sentence"] if include_current else None)
        seq = [vad_lex_text(p) for p in parts]
        feats.append(sequence_to_flow(seq, dim=3))

    feats = np.vstack(feats).astype(np.float32)
    np.save(cache_path, feats)
    return feats

def get_vadbert_logits(texts, cache_name, batch_size=32, max_length=128):
    cache_path = cache_npy_path(cache_name)
    if os.path.exists(cache_path):
        return np.load(cache_path)

    tok = AutoTokenizer.from_pretrained(VADBERT_MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(VADBERT_MODEL_NAME).to(DEVICE)
    model.eval()

    feats = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"VAD-BERT {cache_name}"):
            batch = texts[i:i+batch_size]
            enc = tok(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(DEVICE)
            out = model(**enc).logits
            feats.append(out.detach().cpu().numpy())

    feats = np.vstack(feats).astype(np.float32)
    np.save(cache_path, feats)
    return feats

def emoflow_vadbert_features(df, cache_name, include_current=True):
    cache_path = cache_npy_path(cache_name)
    if os.path.exists(cache_path):
        return np.load(cache_path)

    all_parts = []
    row_parts = []

    for _, r in df.iterrows():
        parts = split_context_sentences(r["context"], r["sentence"] if include_current else None)
        if len(parts) == 0:
            parts = [""]
        row_parts.append(parts)
        all_parts.extend(parts)

    all_logits = get_vadbert_logits(all_parts, cache_name + "_segments")
    cursor = 0
    feats = []

    for parts in row_parts:
        n = len(parts)
        seq = [x for x in all_logits[cursor:cursor+n]]
        cursor += n
        feats.append(sequence_to_flow(seq, dim=all_logits.shape[1]))

    feats = np.vstack(feats).astype(np.float32)
    np.save(cache_path, feats)
    return feats


def eval_multilabel(y_true, y_pred):
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_p": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_r": recall_score(y_true, y_pred, average="micro", zero_division=0),
    }

def train_eval_strategy(run_name, X_train, y_train, X_test, y_test):
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", OneVsRestClassifier(
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="liblinear"
            )
        ))
    ])

    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    metrics = eval_multilabel(y_test, pred)

    model_path = os.path.join(RESULTS_DIR, f"model_{run_name}.joblib")
    dump(clf, model_path)

    return metrics, pred, model_path

def concat_features(*xs):
    return np.concatenate(xs, axis=1).astype(np.float32)

all_results = []
all_reports = {}

for backbone_key, backbone_name in BACKBONES.items():
    print("\n" + "#"*90)
    print("BACKBONE:", backbone_key, "|", backbone_name)
    print("#"*90)

    for task in TASKS:
        print("\n" + "="*80)
        print("TASK:", task)
        print("="*80)

        dev_task, dev_texts, y_dev = build_task(dev, task)
        test_task, test_texts, y_test = build_task(test, task)

        if len(dev_task) < 10 or len(test_task) < 5:
            print("Skipping task because it has too few examples.")
            continue

        include_current = task not in ["forecast_next", "forecast_change_subset"]

        idx_train, idx_val = train_test_split(
            np.arange(len(dev_task)),
            test_size=0.20,
            random_state=42,
            shuffle=True
        )

        prefix_dev = f"{backbone_key}_{task}_dev"
        prefix_test = f"{backbone_key}_{task}_test"

        dev_text_feat = get_transformer_cls_features(
            dev_texts,
            model_name=backbone_name,
            cache_name=row_hash(dev_texts, prefix_dev + "_text"),
            max_length=192
        )

        test_text_feat = get_transformer_cls_features(
            test_texts,
            model_name=backbone_name,
            cache_name=row_hash(test_texts, prefix_test + "_text"),
            max_length=192
        )

        dev_vadlex_flow = emoflow_vadlex_features(
            dev_task,
            cache_name=f"{task}_dev_vadlex_flow_current_{include_current}",
            include_current=include_current
        )
        test_vadlex_flow = emoflow_vadlex_features(
            test_task,
            cache_name=f"{task}_test_vadlex_flow_current_{include_current}",
            include_current=include_current
        )

        dev_vadbert_global = get_vadbert_logits(
            dev_texts,
            cache_name=row_hash(dev_texts, f"{task}_dev_vadbert_global")
        )
        test_vadbert_global = get_vadbert_logits(
            test_texts,
            cache_name=row_hash(test_texts, f"{task}_test_vadbert_global")
        )

        dev_vadbert_flow = emoflow_vadbert_features(
            dev_task,
            cache_name=f"{task}_dev_vadbert_flow_current_{include_current}",
            include_current=include_current
        )
        test_vadbert_flow = emoflow_vadbert_features(
            test_task,
            cache_name=f"{task}_test_vadbert_flow_current_{include_current}",
            include_current=include_current
        )

        strategies = {
            "text_only": (
                dev_text_feat,
                test_text_feat
            ),
            "text_plus_vadlex_emoflow": (
                concat_features(dev_text_feat, dev_vadlex_flow),
                concat_features(test_text_feat, test_vadlex_flow)
            ),
            "text_plus_vadbert_global": (
                concat_features(dev_text_feat, dev_vadbert_global),
                concat_features(test_text_feat, test_vadbert_global)
            ),
            "text_plus_vadbert_emoflow": (
                concat_features(dev_text_feat, dev_vadbert_flow),
                concat_features(test_text_feat, test_vadbert_flow)
            ),
            "text_plus_all_affective": (
                concat_features(
                    dev_text_feat,
                    dev_vadlex_flow,
                    dev_vadbert_global,
                    dev_vadbert_flow
                ),
                concat_features(
                    test_text_feat,
                    test_vadlex_flow,
                    test_vadbert_global,
                    test_vadbert_flow
                )
            ),
        }

        for strat, (X_dev_all, X_test_all) in strategies.items():
            print("\nStrategy:", backbone_key, strat)

            val_name = f"{backbone_key}_{task}_{strat}_val"
            test_name = f"{backbone_key}_{task}_{strat}_test"

            metrics_val, pred_val, _ = train_eval_strategy(
                val_name,
                X_dev_all[idx_train],
                y_dev[idx_train],
                X_dev_all[idx_val],
                y_dev[idx_val]
            )

            metrics_test, pred_test, model_path = train_eval_strategy(
                test_name,
                X_dev_all,
                y_dev,
                X_test_all,
                y_test
            )

            row = {
                "backbone": backbone_key,
                "backbone_model": backbone_name,
                "task": task,
                "strategy": strat,
                "n_dev": len(dev_task),
                "n_test": len(test_task),
                "val_micro_f1": metrics_val["micro_f1"],
                "val_macro_f1": metrics_val["macro_f1"],
                "test_micro_f1": metrics_test["micro_f1"],
                "test_macro_f1": metrics_test["macro_f1"],
                "test_micro_p": metrics_test["micro_p"],
                "test_micro_r": metrics_test["micro_r"],
                "model_path": model_path,
            }

            all_results.append(row)
            print(json.dumps(row, indent=2))

            report = classification_report(
                y_test,
                pred_test,
                target_names=PLUTCHIK,
                zero_division=0,
                output_dict=False
            )
            all_reports[(backbone_key, task, strat)] = report

results_df = pd.DataFrame(all_results).sort_values(
    ["backbone", "task", "test_macro_f1", "test_micro_f1"],
    ascending=[True, True, False, False]
)

results_csv = os.path.join(RESULTS_DIR, "frozen_results.csv")
results_json = os.path.join(RESULTS_DIR, "frozen_classification_reports.json")

results_df.to_csv(results_csv, index=False)

reports_to_save = {
    f"{bb}__{task}__{strat}": report
    for (bb, task, strat), report in all_reports.items()
}

with open(results_json, "w", encoding="utf-8") as f:
    json.dump(reports_to_save, f, indent=2)

print("\nSaved results to:")
print(results_csv)
print(results_json)

display(results_df)


## Five-seed fine-tuning and quantitative analyses

In [ ]:
# ============================================================
# StoryCommonsense EmoFlow fine-tuning
# BERT-tiny
# Real multi-seed fine-tuning + paired bootstrap + per-emotion deltas
# ============================================================


import os, re, ast, json, hashlib, warnings, random, gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")

# -----------------------------
# Config
# -----------------------------

PROJECT_DIR = os.path.abspath(os.environ.get("EMOFLOW_PROJECT_DIR", "."))
DATA_DIR = os.path.abspath(os.environ.get("EMOFLOW_DATA_DIR", os.path.join(PROJECT_DIR, "data")))
TRAIN_PATH = os.path.join(DATA_DIR, "storycs_train.csv")
DEV_PATH   = os.path.join(DATA_DIR, "storycs_dev.csv")
TEST_PATH  = os.path.join(DATA_DIR, "storycs_test.csv")

CACHE_DIR = os.path.join(PROJECT_DIR, "cache", "finetuning")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results", "finetuning")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

VAD_LEXICON_PATH = os.path.abspath(
    os.environ.get(
        "NRC_VAD_PATH",
        os.path.join(DATA_DIR, "unigrams-NRC-VAD-Lexicon-v2.1.txt")
    )
)

required_files = [TRAIN_PATH, DEV_PATH, TEST_PATH, VAD_LEXICON_PATH]
missing_files = [path for path in required_files if not os.path.exists(path)]
if missing_files:
    raise FileNotFoundError(
        "Missing required input files. See README.md:\n" + "\n".join(missing_files)
    )
PLUTCHIK = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]

TEXT_MODEL_NAME = "prajjwal1/bert-tiny"
TEXT_MODEL_KEY = "bert_tiny"
VADBERT_MODEL_NAME = "RobroKools/vad-bert"

SEEDS = [12, 24, 42, 73, 105]
#SEEDS = [42]


TASKS = {
    "current": "state_current",
    "forecast": "forecast_next",
    "transition": "change_subset",
    "forecast-transition": "forecast_change_subset",
}

STRATEGIES = [
    "text_only",
    "text_plus_vadlex_emoflow",
    "text_plus_vadbert_global",
    "text_plus_vadbert_emoflow",
    "text_plus_all_affective",
]

MAX_LEN = 192
BATCH_SIZE = 128
EPOCHS = 10
LR = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
BOOTSTRAP_SAMPLES = 3000
THRESHOLD = 0.5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# -----------------------------
# Repro
# -----------------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -----------------------------
# Load data
# -----------------------------

train_raw = pd.read_csv(TRAIN_PATH)
dev_raw   = pd.read_csv(DEV_PATH)
test_raw  = pd.read_csv(TEST_PATH)

print("train:", train_raw.shape)
print("dev:", dev_raw.shape)
print("test:", test_raw.shape)

# -----------------------------
# Helpers
# -----------------------------

def safe_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        y = ast.literal_eval(str(x))
        return y if isinstance(y, list) else [str(y)]
    except Exception:
        return [str(x)]

def parse_plutchik(x):
    vals = np.zeros(len(PLUTCHIK), dtype=np.float32)
    for item in safe_list(x):
        s = str(item).strip().lower()
        if s == "none":
            continue
        if ":" in s:
            lab, score = s.split(":", 1)
            lab = lab.strip()
            try:
                score = float(score)
            except Exception:
                score = 1.0
        else:
            lab, score = s.strip(), 3.0
        if lab in PLUTCHIK:
            vals[PLUTCHIK.index(lab)] = max(vals[PLUTCHIK.index(lab)], score / 3.0)
    return vals

def binarize_y(y, threshold=1e-6):
    return (y > threshold).astype(int)

def clean_text(x):
    return "" if pd.isna(x) else str(x)

def make_full_text(row):
    ctx = clean_text(row.get("context", ""))
    sent = clean_text(row.get("sentence", ""))
    return (ctx.strip() + " " + sent.strip()).strip() if ctx.strip() else sent.strip()

def make_prev_text(row):
    return clean_text(row.get("context", "")).strip()

def cache_npy_path(name):
    safe = re.sub(r"[^a-zA-Z0-9_\-\.]+", "_", name)
    return os.path.join(CACHE_DIR, safe + ".npy")

def row_hash(texts, prefix):
    h = hashlib.md5()
    for t in texts:
        h.update(str(t).encode("utf-8", errors="ignore"))
    return prefix + "_" + h.hexdigest()[:16]

def split_context_sentences(context, current_sentence=None):
    parts = []
    ctx = clean_text(context)

    if ctx.strip():
        rough = re.split(r"(?<=[\.\!\?])\s+", ctx)
        if len(rough) <= 1:
            rough = ctx.replace("!", ".").replace("?", ".").split(".")
        parts += [p.strip() for p in rough if p.strip()]

    if current_sentence is not None and clean_text(current_sentence).strip():
        parts.append(clean_text(current_sentence).strip())

    return parts

# -----------------------------
# Aggregate annotations
# -----------------------------

def aggregate_annotations(df):
    df = df.copy()
    df["full_text"] = df.apply(make_full_text, axis=1)
    df["prev_text"] = df.apply(make_prev_text, axis=1)
    df["plutchik_vec"] = df["plutchik"].apply(parse_plutchik)

    rows = []
    group_cols = ["storyid", "linenum", "char"]

    for key, g in df.groupby(group_cols, sort=False):
        y = np.vstack(g["plutchik_vec"].values).mean(axis=0)
        first = g.iloc[0]
        rows.append({
            "storyid": key[0],
            "linenum": key[1],
            "char": key[2],
            "sentence": clean_text(first["sentence"]),
            "context": clean_text(first["context"]),
            "full_text": clean_text(first["full_text"]),
            "prev_text": clean_text(first["prev_text"]),
            "affected_yes_ratio": np.mean(g["affected"].astype(str).str.lower().eq("yes")),
            "y": y,
        })

    out = pd.DataFrame(rows)
    return out.sort_values(["storyid", "char", "linenum"]).reset_index(drop=True)

def add_previous_and_story_dynamics(df):
    df = df.copy()

    prevs = []
    transition_counts = {}
    accumulated_distances = {}
    story_lengths = {}

    for (storyid, char), g in df.groupby(["storyid", "char"], sort=False):
        g = g.sort_values("linenum")
        ys = list(g["y"].values)
        idxs = list(g.index)

        y_bin = [binarize_y(y.reshape(1, -1))[0] for y in ys]

        n_trans = 0
        acc_dist = 0.0

        for pos, idx in enumerate(idxs):
            if pos == 0:
                prevs.append((idx, np.zeros(len(PLUTCHIK), dtype=np.float32), False))
            else:
                prevs.append((idx, ys[pos - 1], True))
                if np.any(y_bin[pos] != y_bin[pos - 1]):
                    n_trans += 1
                acc_dist += float(np.linalg.norm(ys[pos] - ys[pos - 1], ord=1))

        for idx in idxs:
            transition_counts[idx] = n_trans
            accumulated_distances[idx] = acc_dist
            story_lengths[idx] = len(g)

    prev_map = {idx: (vec, has_prev) for idx, vec, has_prev in prevs}

    df["y_prev"] = df.index.map(lambda i: prev_map[i][0])
    df["has_prev"] = df.index.map(lambda i: prev_map[i][1])
    df["changed_from_prev"] = df.apply(
        lambda r: bool(np.any(binarize_y(r["y"].reshape(1, -1))[0] != binarize_y(r["y_prev"].reshape(1, -1))[0])) if r["has_prev"] else False,
        axis=1
    )

    df["n_story_char_lines"] = df.index.map(story_lengths)
    df["n_character_transitions"] = df.index.map(transition_counts)
    df["accum_emotional_distance"] = df.index.map(accumulated_distances)
    df["n_context_sentences"] = df.apply(lambda r: len(split_context_sentences(r["context"], None)), axis=1)
    df["n_full_sentences"] = df.apply(lambda r: len(split_context_sentences(r["context"], r["sentence"])), axis=1)
    df["n_full_words"] = df["full_text"].apply(lambda x: len(str(x).split()))
    df["n_prev_words"] = df["prev_text"].apply(lambda x: len(str(x).split()))

    return df

dev = add_previous_and_story_dynamics(aggregate_annotations(dev_raw))
test = add_previous_and_story_dynamics(aggregate_annotations(test_raw))

print("Aggregated dev:", dev.shape)
print("Aggregated test:", test.shape)

# -----------------------------
# Task builder
# -----------------------------

def build_task(df, task_name):
    df = df.copy()

    if task_name == "state_current":
        df["input_text"] = df.apply(
            lambda r: f"Character: {r['char']}. Story: {r['full_text']}",
            axis=1
        )
        use = df.copy()

    elif task_name == "forecast_next":
        use = df[df["has_prev"] & df["prev_text"].astype(str).str.len().gt(0)].copy()
        use["input_text"] = use.apply(
            lambda r: f"Character: {r['char']}. Previous story: {r['prev_text']}",
            axis=1
        )

    elif task_name == "change_subset":
        use = df[df["has_prev"] & df["changed_from_prev"]].copy()
        use["input_text"] = use.apply(
            lambda r: f"Character: {r['char']}. Story: {r['full_text']}",
            axis=1
        )

    elif task_name == "forecast_change_subset":
        use = df[
            df["has_prev"] &
            df["changed_from_prev"] &
            df["prev_text"].astype(str).str.len().gt(0)
        ].copy()
        use["input_text"] = use.apply(
            lambda r: f"Character: {r['char']}. Previous story: {r['prev_text']}",
            axis=1
        )

    else:
        raise ValueError(f"Unknown task: {task_name}")

    use = use.reset_index(drop=True)
    texts = use["input_text"].tolist()
    y = binarize_y(np.vstack(use["y"].values))

    return use, texts, y

# -----------------------------
# VAD lexicon and affective features
# -----------------------------

def load_nrc_vad(path):
    lex = {}
    if not os.path.exists(path):
        print("NRC VAD lexicon not found. VAD-lex features will be zeros:", path)
        return lex

    tmp = pd.read_csv(path, sep="\t")
    tmp.columns = [c.lower() for c in tmp.columns]

    word_col = "word" if "word" in tmp.columns else tmp.columns[0]
    v_col = "valence" if "valence" in tmp.columns else tmp.columns[1]
    a_col = "arousal" if "arousal" in tmp.columns else tmp.columns[2]
    d_col = "dominance" if "dominance" in tmp.columns else tmp.columns[3]

    for _, r in tmp.iterrows():
        lex[str(r[word_col]).lower()] = np.array([float(r[v_col]), float(r[a_col]), float(r[d_col])], dtype=np.float32)

    print("Loaded NRC VAD entries:", len(lex))
    return lex

VAD_LEX = load_nrc_vad(VAD_LEXICON_PATH)

def simple_tokenize(text):
    return [w.strip(".,;:!?()[]{}\"'").lower() for w in str(text).split()]

def vad_lex_text(text):
    vals = [VAD_LEX[w] for w in simple_tokenize(text) if w in VAD_LEX]
    if not vals:
        return np.zeros(3, dtype=np.float32)
    return np.vstack(vals).mean(axis=0).astype(np.float32)

def sequence_to_flow(seq, dim):
    if len(seq) == 0:
        seq = [np.zeros(dim, dtype=np.float32)]

    seq = [np.asarray(x, dtype=np.float32) for x in seq]

    while len(seq) < 3:
        seq.insert(0, np.zeros(dim, dtype=np.float32))

    states = seq[-3:]
    deltas = [states[1] - states[0], states[2] - states[1]]
    global_state = np.vstack(seq).mean(axis=0)

    return np.concatenate(states + deltas + [global_state], axis=0).astype(np.float32)

def emoflow_vadlex_features(df, cache_name, include_current=True):
    cache_path = cache_npy_path(cache_name)
    if os.path.exists(cache_path):
        return np.load(cache_path)

    feats = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"VAD-lex flow {cache_name}"):
        parts = split_context_sentences(r["context"], r["sentence"] if include_current else None)
        seq = [vad_lex_text(p) for p in parts]
        feats.append(sequence_to_flow(seq, dim=3))

    feats = np.vstack(feats).astype(np.float32)
    np.save(cache_path, feats)
    return feats

def get_vadbert_logits(texts, cache_name, batch_size=64, max_length=128):
    cache_path = cache_npy_path(cache_name)
    if os.path.exists(cache_path):
        return np.load(cache_path)

    tok = AutoTokenizer.from_pretrained(VADBERT_MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(VADBERT_MODEL_NAME).to(DEVICE)
    model.eval()

    feats = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"VAD-BERT {cache_name}"):
            batch = texts[i:i+batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(DEVICE)
            out = model(**enc).logits
            feats.append(out.detach().cpu().numpy())

    feats = np.vstack(feats).astype(np.float32)
    np.save(cache_path, feats)
    return feats

def emoflow_vadbert_features(df, cache_name, include_current=True):
    cache_path = cache_npy_path(cache_name)
    if os.path.exists(cache_path):
        return np.load(cache_path)

    all_parts = []
    row_parts = []

    for _, r in df.iterrows():
        parts = split_context_sentences(r["context"], r["sentence"] if include_current else None)
        if len(parts) == 0:
            parts = [""]
        row_parts.append(parts)
        all_parts.extend(parts)

    all_logits = get_vadbert_logits(all_parts, cache_name + "_segments")

    cursor = 0
    feats = []

    for parts in row_parts:
        n = len(parts)
        seq = [x for x in all_logits[cursor:cursor+n]]
        cursor += n
        feats.append(sequence_to_flow(seq, dim=all_logits.shape[1]))

    feats = np.vstack(feats).astype(np.float32)
    np.save(cache_path, feats)
    return feats

def concat_features(*xs):
    return np.concatenate(xs, axis=1).astype(np.float32)

# -----------------------------
# Dataset and model
# -----------------------------

class StoryAffectDataset(Dataset):
    def __init__(self, texts, y, affect_features, tokenizer, max_len=192):
        self.texts = texts
        self.y = y.astype(np.float32)
        self.affect_features = affect_features.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "affect": torch.tensor(self.affect_features[idx], dtype=torch.float32),
            "labels": torch.tensor(self.y[idx], dtype=torch.float32),
        }
        if "token_type_ids" in enc:
            item["token_type_ids"] = enc["token_type_ids"].squeeze(0)
        return item

def compute_pos_weight(y_train):
    positives = y_train.sum(axis=0)
    negatives = y_train.shape[0] - positives
    pos_weight = negatives / np.maximum(positives, 1.0)
    pos_weight = np.clip(pos_weight, 1.0, 20.0)
    return torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)


def tune_thresholds(y_true, probs, thresholds=np.arange(0.05, 0.76, 0.05)):
    best_thresholds = []

    for j in range(y_true.shape[1]):
        best_t = 0.5
        best_f1 = -1

        for t in thresholds:
            pred = (probs[:, j] >= t).astype(int)
            f1 = f1_score(y_true[:, j], pred, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresholds.append(best_t)

    return np.array(best_thresholds, dtype=np.float32)


def probs_to_pred(probs, thresholds):
    return (probs >= thresholds.reshape(1, -1)).astype(int)

class BertTinyWithAffect(nn.Module):
    def __init__(self, model_name, affect_dim, num_labels=8, dropout=0.1, pos_weight=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden + affect_dim, num_labels)
        self.pos_weight = pos_weight

    def forward(self, input_ids, attention_mask, affect, token_type_ids=None, labels=None):
        kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        out = self.encoder(**kwargs)
        cls = out.last_hidden_state[:, 0, :]
        x = torch.cat([cls, affect], dim=1)
        x = self.dropout(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

# -----------------------------
# Metrics
# -----------------------------

def eval_multilabel(y_true, y_pred):
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_p": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_r": recall_score(y_true, y_pred, average="micro", zero_division=0),
    }

def per_emotion_f1(y_true, y_pred):
    return {
        lab: f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
        for i, lab in enumerate(PLUTCHIK)
    }

def paired_bootstrap_macro_f1(y_true, pred_a, pred_b, n_boot=3000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs = np.zeros(n_boot, dtype=np.float32)

    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        fa = f1_score(y_true[idx], pred_a[idx], average="macro", zero_division=0)
        fb = f1_score(y_true[idx], pred_b[idx], average="macro", zero_division=0)
        diffs[b] = fb - fa

    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_two_sided = 2 * min(np.mean(diffs <= 0), np.mean(diffs >= 0))
    p_two_sided = min(float(p_two_sided), 1.0)

    return {
        "delta_macro_f1_mean": float(np.mean(diffs)),
        "delta_macro_f1_ci_low": float(ci_low),
        "delta_macro_f1_ci_high": float(ci_high),
        "p_two_sided": p_two_sided,
    }

# -----------------------------
# Fine-tuning and prediction
# -----------------------------

def make_loader(dataset, batch_size, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0
    )

def predict_model(model, loader):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("labels")
            out = model(**batch)
            probs = torch.sigmoid(out["logits"]).detach().cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.detach().cpu().numpy())

    probs = np.vstack(all_probs)
    labels = np.vstack(all_labels).astype(int)

    return probs, labels

def train_one_model(seed, strategy, task_name, train_texts, train_y, train_affect, val_texts, val_y, val_affect, test_texts, test_y, test_affect):
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

    train_ds = StoryAffectDataset(train_texts, train_y, train_affect, tokenizer, max_len=MAX_LEN)
    val_ds   = StoryAffectDataset(val_texts, val_y, val_affect, tokenizer, max_len=MAX_LEN)
    test_ds  = StoryAffectDataset(test_texts, test_y, test_affect, tokenizer, max_len=MAX_LEN)

    train_loader = make_loader(train_ds, BATCH_SIZE, shuffle=True, seed=seed)
    val_loader   = make_loader(val_ds, BATCH_SIZE, shuffle=False, seed=seed)
    test_loader  = make_loader(test_ds, BATCH_SIZE, shuffle=False, seed=seed)

    pos_weight = compute_pos_weight(train_y)

    model = BertTinyWithAffect(
        TEXT_MODEL_NAME,
        affect_dim=train_affect.shape[1],
        num_labels=len(PLUTCHIK),
        pos_weight=pos_weight
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_val_macro = -1
    best_state = None

    PATIENCE = 2
    bad_epochs = 0


    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader, desc=f"{task_name} | {strategy} | seed {seed} | epoch {epoch}"):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            loss = out["loss"]

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

        val_probs, val_labels = predict_model(model, val_loader)
        val_thresholds = tune_thresholds(val_labels, val_probs)
        val_pred = probs_to_pred(val_probs, val_thresholds)
        val_metrics = eval_multilabel(val_labels, val_pred)

        print({
            "epoch": epoch,
            "train_loss": total_loss / max(len(train_loader), 1),
            "val_macro_f1": val_metrics["macro_f1"],
            "val_micro_f1": val_metrics["micro_f1"],
        })


        if val_metrics["macro_f1"] > best_val_macro + 1e-4:
            best_val_macro = val_metrics["macro_f1"]
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break


    model.load_state_dict(best_state)

    val_probs, val_labels = predict_model(model, val_loader)
    best_thresholds = tune_thresholds(val_labels, val_probs)

    test_probs, test_labels = predict_model(model, test_loader)
    test_pred = probs_to_pred(test_probs, best_thresholds)

    test_metrics = eval_multilabel(test_labels, test_pred)

    model_path = os.path.join(RESULTS_DIR, f"finetuned_{TEXT_MODEL_KEY}_{task_name}_{strategy}_seed{seed}.pt")
    torch.save(model.state_dict(), model_path)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return val_metrics, test_metrics, test_pred, test_probs, model_path

def safe_qcut_labels(series, q=3, labels3=("low", "medium", "high")):
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(index=series.index, dtype="object")

    valid = s.dropna()

    if valid.nunique() <= 1:
        out.loc[:] = "single"
        return out.astype("category")

    try:
        codes, bins = pd.qcut(
            valid,
            q=q,
            labels=False,
            retbins=True,
            duplicates="drop"
        )

        codes = pd.Series(codes, index=valid.index)
        unique_codes = sorted(codes.dropna().unique())
        n_bins = len(unique_codes)

        if n_bins <= 1:
            out.loc[:] = "single"
            return out.astype("category")

        if n_bins == 2:
            labels = [labels3[0], labels3[-1]]
        elif n_bins == 3:
            labels = list(labels3)
        else:
            labels = [f"bin_{i+1}" for i in range(n_bins)]

        code_to_label = {code: labels[i] for i, code in enumerate(unique_codes)}
        out.loc[valid.index] = codes.map(code_to_label)
        out.loc[s.isna()] = "missing"

        return out.astype("category")

    except Exception as e:
        print("safe_qcut_labels fallback:", repr(e))
        out.loc[:] = "single"
        return out.astype("category")


# -----------------------------
# Main experiment
# -----------------------------

all_rows = []
all_pred_records = {}
per_emotion_rows = []
analysis_rows = []

for display_task, task_name in TASKS.items():

    dev_task, dev_texts, y_dev = build_task(dev, task_name)
    test_task, test_texts, y_test = build_task(test, task_name)

    print("\n" + "="*70)
    print(display_task)
    print("="*70)

    counts = y_test.sum(axis=0)

    df_counts = pd.DataFrame({
        "emotion": PLUTCHIK,
        "positive_examples": counts.astype(int),
        "percentage": (counts / len(y_test) * 100).round(2)
    }).sort_values("positive_examples", ascending=False)

    print(df_counts)

for display_task, task_name in TASKS.items():
    print("\n" + "="*90)
    print("TASK:", display_task, "|", task_name)
    print("="*90)

    dev_task, dev_texts, y_dev = build_task(dev, task_name)
    test_task, test_texts, y_test = build_task(test, task_name)

    include_current = task_name not in ["forecast_next", "forecast_change_subset"]

    print("n_dev:", len(dev_task), "n_test:", len(test_task), "include_current:", include_current)

    zeros_dev = np.zeros((len(dev_task), 0), dtype=np.float32)
    zeros_test = np.zeros((len(test_task), 0), dtype=np.float32)

    dev_vadlex_flow = emoflow_vadlex_features(
        dev_task,
        cache_name=f"{task_name}_dev_vadlex_flow_current_{include_current}",
        include_current=include_current
    )
    test_vadlex_flow = emoflow_vadlex_features(
        test_task,
        cache_name=f"{task_name}_test_vadlex_flow_current_{include_current}",
        include_current=include_current
    )

    dev_vadbert_global = get_vadbert_logits(
        dev_texts,
        cache_name=row_hash(dev_texts, f"{task_name}_dev_vadbert_global")
    )
    test_vadbert_global = get_vadbert_logits(
        test_texts,
        cache_name=row_hash(test_texts, f"{task_name}_test_vadbert_global")
    )

    dev_vadbert_flow = emoflow_vadbert_features(
        dev_task,
        cache_name=f"{task_name}_dev_vadbert_flow_current_{include_current}",
        include_current=include_current
    )
    test_vadbert_flow = emoflow_vadbert_features(
        test_task,
        cache_name=f"{task_name}_test_vadbert_flow_current_{include_current}",
        include_current=include_current
    )

    affect_sets = {
        "text_only": (
            zeros_dev,
            zeros_test
        ),
        "text_plus_vadlex_emoflow": (
            dev_vadlex_flow,
            test_vadlex_flow
        ),
        "text_plus_vadbert_global": (
            dev_vadbert_global,
            test_vadbert_global
        ),
        "text_plus_vadbert_emoflow": (
            dev_vadbert_flow,
            test_vadbert_flow
        ),
        "text_plus_all_affective": (
            concat_features(dev_vadlex_flow, dev_vadbert_global, dev_vadbert_flow),
            concat_features(test_vadlex_flow, test_vadbert_global, test_vadbert_flow)
        ),
    }

    for strategy in STRATEGIES:
        print("\nStrategy:", display_task, strategy)

        dev_affect_all, test_affect = affect_sets[strategy]
        seed_preds = []
        seed_probs = []

        for seed in SEEDS:
            set_seed(seed)

            idx_train, idx_val = train_test_split(
                np.arange(len(dev_task)),
                test_size=0.20,
                random_state=seed,
                shuffle=True
            )

            train_texts = [dev_texts[i] for i in idx_train]
            val_texts = [dev_texts[i] for i in idx_val]

            train_y = y_dev[idx_train]
            val_y = y_dev[idx_val]

            train_affect = dev_affect_all[idx_train]
            val_affect = dev_affect_all[idx_val]

            val_metrics, test_metrics, pred_test, probs_test, model_path = train_one_model(
                seed=seed,
                strategy=strategy,
                task_name=task_name,
                train_texts=train_texts,
                train_y=train_y,
                train_affect=train_affect,
                val_texts=val_texts,
                val_y=val_y,
                val_affect=val_affect,
                test_texts=test_texts,
                test_y=y_test,
                test_affect=test_affect,
            )

            seed_preds.append(pred_test)
            seed_probs.append(probs_test)

            row = {
                "task_display": display_task,
                "task": task_name,
                "strategy": strategy,
                "seed": seed,
                "n_dev": len(dev_task),
                "n_test": len(test_task),
                "include_current": include_current,
                "val_micro_f1": val_metrics["micro_f1"],
                "val_macro_f1": val_metrics["macro_f1"],
                "test_micro_f1": test_metrics["micro_f1"],
                "test_macro_f1": test_metrics["macro_f1"],
                "test_micro_p": test_metrics["micro_p"],
                "test_micro_r": test_metrics["micro_r"],
                "model_path": model_path,
            }
            all_rows.append(row)
            print(json.dumps(row, indent=2))

            emo_f1 = per_emotion_f1(y_test, pred_test)
            for emo, val in emo_f1.items():
                per_emotion_rows.append({
                    "task_display": display_task,
                    "task": task_name,
                    "strategy": strategy,
                    "seed": seed,
                    "emotion": emo,
                    "f1": val,
                })

        all_pred_records[(display_task, strategy)] = {
            "y_test": y_test,
            "seed_preds": seed_preds,
            "seed_probs": seed_probs,
            "test_task": test_task.copy(),
        }

    # Analysis by length, transition count, and accumulated emotional distance.
    for strategy in STRATEGIES:
        rec = all_pred_records[(display_task, strategy)]
        pred_stack = np.stack(rec["seed_preds"], axis=0)
        pred_majority = (pred_stack.mean(axis=0) >= 0.5).astype(int)
        y_true = rec["y_test"]
        meta = rec["test_task"].copy()

        length_col = "n_full_words" if include_current else "n_prev_words"

        meta["length_bin"] = safe_qcut_labels(
            meta[length_col],
            q=3,
            labels3=("short", "medium", "long")
        )

        meta["transition_bin"] = safe_qcut_labels(
            meta["n_character_transitions"],
            q=3,
            labels3=("low", "medium", "high")
        )

        meta["distance_bin"] = safe_qcut_labels(
            meta["accum_emotional_distance"],
            q=3,
            labels3=("low", "medium", "high")
        )

        for factor in ["length_bin", "transition_bin", "distance_bin"]:
            for bin_value in meta[factor].dropna().unique():
                idx = meta.index[meta[factor] == bin_value].to_numpy()
                if len(idx) < 10:
                    continue
                metrics = eval_multilabel(y_true[idx], pred_majority[idx])
                analysis_rows.append({
                    "task_display": display_task,
                    "task": task_name,
                    "strategy": strategy,
                    "factor": factor,
                    "bin": str(bin_value),
                    "n": len(idx),
                    "micro_f1": metrics["micro_f1"],
                    "macro_f1": metrics["macro_f1"],
                    "micro_p": metrics["micro_p"],
                    "micro_r": metrics["micro_r"],
                })

# -----------------------------
# Save raw results
# -----------------------------

results_df = pd.DataFrame(all_rows)
results_path = os.path.join(RESULTS_DIR, "finetune_seed_results_bert_tiny_no_senticnet.csv")
results_df.to_csv(results_path, index=False)

per_emotion_df = pd.DataFrame(per_emotion_rows)
per_emotion_path = os.path.join(RESULTS_DIR, "finetune_per_emotion_f1_bert_tiny_no_senticnet.csv")
per_emotion_df.to_csv(per_emotion_path, index=False)

analysis_df = pd.DataFrame(analysis_rows)
analysis_path = os.path.join(RESULTS_DIR, "finetune_length_transition_distance_analysis_bert_tiny_no_senticnet.csv")
analysis_df.to_csv(analysis_path, index=False)

# -----------------------------
# Compact summary
# -----------------------------

summary_df = (
    results_df
    .groupby(["task_display", "task", "strategy"], as_index=False)
    .agg(
        n_test=("n_test", "first"),
        test_micro_f1_mean=("test_micro_f1", "mean"),
        test_micro_f1_std=("test_micro_f1", "std"),
        test_macro_f1_mean=("test_macro_f1", "mean"),
        test_macro_f1_std=("test_macro_f1", "std"),
        test_micro_p_mean=("test_micro_p", "mean"),
        test_micro_r_mean=("test_micro_r", "mean"),
    )
)

baseline_macro = summary_df[summary_df["strategy"] == "text_only"][
    ["task_display", "test_macro_f1_mean"]
].rename(columns={"test_macro_f1_mean": "baseline_macro_f1"})

summary_df = summary_df.merge(baseline_macro, on="task_display", how="left")
summary_df["delta_vs_text_macro_f1"] = summary_df["test_macro_f1_mean"] - summary_df["baseline_macro_f1"]

summary_df["task_display"] = pd.Categorical(
    summary_df["task_display"],
    categories=["current", "forecast", "transition", "forecast-transition"],
    ordered=True
)

summary_df = summary_df.sort_values(
    ["task_display", "test_macro_f1_mean"],
    ascending=[True, False]
)

summary_path = os.path.join(RESULTS_DIR, "finetune_compact_summary_bert_tiny_no_senticnet.csv")
summary_df.to_csv(summary_path, index=False)

# -----------------------------
# Bootstrap paired vs baseline
# -----------------------------

bootstrap_rows = []

for display_task in TASKS.keys():
    y_true = all_pred_records[(display_task, "text_only")]["y_test"]

    for seed_idx, seed in enumerate(SEEDS):
        baseline_pred = all_pred_records[(display_task, "text_only")]["seed_preds"][seed_idx]

        for strategy in STRATEGIES:
            if strategy == "text_only":
                continue

            pred = all_pred_records[(display_task, strategy)]["seed_preds"][seed_idx]

            bs = paired_bootstrap_macro_f1(
                y_true,
                baseline_pred,
                pred,
                n_boot=BOOTSTRAP_SAMPLES,
                seed=seed
            )

            bootstrap_rows.append({
                "task_display": display_task,
                "strategy": strategy,
                "seed": seed,
                **bs
            })

bootstrap_df = pd.DataFrame(bootstrap_rows)

bootstrap_summary_df = (
    bootstrap_df
    .groupby(["task_display", "strategy"], as_index=False)
    .agg(
        delta_macro_f1_mean=("delta_macro_f1_mean", "mean"),
        delta_macro_f1_std=("delta_macro_f1_mean", "std"),
        ci_low_mean=("delta_macro_f1_ci_low", "mean"),
        ci_high_mean=("delta_macro_f1_ci_high", "mean"),
        p_two_sided_median=("p_two_sided", "median"),
    )
)

bootstrap_path = os.path.join(RESULTS_DIR, "finetune_bootstrap_vs_text_bert_tiny_no_senticnet.csv")
bootstrap_summary_path = os.path.join(RESULTS_DIR, "finetune_bootstrap_vs_text_summary_bert_tiny_no_senticnet.csv")

bootstrap_df.to_csv(bootstrap_path, index=False)
bootstrap_summary_df.to_csv(bootstrap_summary_path, index=False)

# -----------------------------
# Per-emotion deltas
# -----------------------------

baseline_emo = per_emotion_df[per_emotion_df["strategy"] == "text_only"][
    ["task_display", "seed", "emotion", "f1"]
].rename(columns={"f1": "baseline_f1"})

per_emotion_delta_df = per_emotion_df.merge(
    baseline_emo,
    on=["task_display", "seed", "emotion"],
    how="left"
)

per_emotion_delta_df["delta_vs_text_f1"] = per_emotion_delta_df["f1"] - per_emotion_delta_df["baseline_f1"]

per_emotion_summary_df = (
    per_emotion_delta_df
    .groupby(["task_display", "strategy", "emotion"], as_index=False)
    .agg(
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        delta_vs_text_f1_mean=("delta_vs_text_f1", "mean"),
        delta_vs_text_f1_std=("delta_vs_text_f1", "std"),
    )
    .sort_values(["task_display", "strategy", "delta_vs_text_f1_mean"], ascending=[True, True, False])
)

per_emotion_delta_path = os.path.join(RESULTS_DIR, "finetune_per_emotion_delta_bert_tiny_no_senticnet.csv")
per_emotion_summary_path = os.path.join(RESULTS_DIR, "finetune_per_emotion_delta_summary_bert_tiny_no_senticnet.csv")

per_emotion_delta_df.to_csv(per_emotion_delta_path, index=False)
per_emotion_summary_df.to_csv(per_emotion_summary_path, index=False)

# -----------------------------
# Show outputs
# -----------------------------

print("\nSaved files:")
print(results_path)
print(summary_path)
print(bootstrap_path)
print(bootstrap_summary_path)
print(per_emotion_path)
print(per_emotion_delta_path)
print(per_emotion_summary_path)
print(analysis_path)

print("\nCompact summary:")
display(summary_df)

print("\nBootstrap paired vs text_only:")
display(bootstrap_summary_df.sort_values(["task_display", "delta_macro_f1_mean"], ascending=[True, False]))

print("\nPer-emotion delta summary:")
display(per_emotion_summary_df)

print("\nLength / transition / emotional-distance analysis:")
display(analysis_df)


## Qualitative error analysis

In [ ]:
# Qualitative comparison used for the error analysis
# Run after the fine-tuning cell. Predictions are combined by majority vote
# across the five seeds, separately for each task.

def labels_from_vector(vector):
    labels = [PLUTCHIK[i] for i, value in enumerate(vector) if int(value) == 1]
    return "|".join(labels) if labels else "none"


def label_set_f1(gold, prediction):
    gold = np.asarray(gold, dtype=int)
    prediction = np.asarray(prediction, dtype=int)
    tp = int(np.sum((gold == 1) & (prediction == 1)))
    fp = int(np.sum((gold == 0) & (prediction == 1)))
    fn = int(np.sum((gold == 1) & (prediction == 0)))
    denominator = 2 * tp + fp + fn
    return 1.0 if denominator == 0 else (2 * tp) / denominator


qualitative_rows = []
for task_display, task_name in TASKS.items():
    text_record = all_pred_records[(task_display, "text_only")]
    flow_record = all_pred_records[(task_display, "text_plus_vadbert_emoflow")]
    y_true = np.asarray(text_record["y_test"], dtype=int)
    metadata = text_record["test_task"].reset_index(drop=True)

    text_stack = np.stack(text_record["seed_preds"], axis=0).astype(int)
    flow_stack = np.stack(flow_record["seed_preds"], axis=0).astype(int)
    text_majority = (text_stack.mean(axis=0) >= 0.5).astype(int)
    flow_majority = (flow_stack.mean(axis=0) >= 0.5).astype(int)

    for i, gold in enumerate(y_true):
        text_prediction = text_majority[i]
        flow_prediction = flow_majority[i]
        text_f1 = label_set_f1(gold, text_prediction)
        flow_f1 = label_set_f1(gold, flow_prediction)
        text_exact = bool(np.array_equal(gold, text_prediction))
        flow_exact = bool(np.array_equal(gold, flow_prediction))

        if flow_exact and not text_exact:
            category = "flow_exact_text_wrong"
        elif text_exact and not flow_exact:
            category = "text_exact_flow_wrong"
        elif flow_f1 > text_f1:
            category = "flow_improves"
        elif flow_f1 < text_f1:
            category = "text_improves"
        elif text_exact:
            category = "both_exact"
        else:
            category = "both_wrong_same_f1"

        row = metadata.iloc[i]
        qualitative_rows.append({
            "task": task_name,
            "instance_index": i,
            "storyid": row.get("storyid", ""),
            "linenum": row.get("linenum", ""),
            "character": row.get("char", ""),
            "model_input": row.get("input_text", ""),
            "target_sentence": row.get("sentence", ""),
            "gold_labels": labels_from_vector(gold),
            "text_prediction": labels_from_vector(text_prediction),
            "flow_prediction": labels_from_vector(flow_prediction),
            "text_label_set_f1": text_f1,
            "flow_label_set_f1": flow_f1,
            "flow_minus_text_f1": flow_f1 - text_f1,
            "category": category,
        })

qualitative_cases = pd.DataFrame(qualitative_rows)
qualitative_cases_path = os.path.join(RESULTS_DIR, "qualitative_cases_all_tasks.csv")
qualitative_cases.to_csv(qualitative_cases_path, index=False)

selected_parts = []
for task_name in TASKS.values():
    task_cases = qualitative_cases[qualitative_cases["task"] == task_name]
    for category in ["flow_exact_text_wrong", "flow_improves", "both_wrong_same_f1"]:
        subset = task_cases[task_cases["category"] == category].copy()
        ascending = category == "both_wrong_same_f1"
        subset = subset.sort_values("flow_minus_text_f1", ascending=ascending)
        selected_parts.append(subset.head(5))

qualitative_examples = pd.concat(selected_parts, ignore_index=True)
qualitative_examples_path = os.path.join(RESULTS_DIR, "qualitative_examples_for_review.csv")
qualitative_examples.to_csv(qualitative_examples_path, index=False)

display(qualitative_cases.groupby(["task", "category"]).size().rename("n").reset_index())
print("Saved:", qualitative_cases_path)
print("Saved:", qualitative_examples_path)


## NRC-VAD coverage

In [ ]:
# NRC-VAD coverage reported in the manuscript
# This preserves the exact post-hoc token definition used for Table 2.

def coverage_tokens(text):
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", str(text).lower())


def coverage_task_dataframe(dataframe, task_name):
    if task_name == "state_current":
        use = dataframe.copy()
    elif task_name == "forecast_next":
        use = dataframe[dataframe["has_prev"].astype(bool)].copy()
    elif task_name == "forecast_change_subset":
        use = dataframe[
            dataframe["has_prev"].astype(bool)
            & dataframe["changed_from_prev"].astype(bool)
        ].copy()
    else:
        raise ValueError(f"Unknown task: {task_name}")
    return use.reset_index(drop=True)


def calculate_vad_coverage(dataframe, task_name, split_name):
    use = coverage_task_dataframe(dataframe, task_name)
    include_current = task_name == "state_current"
    total_tokens = covered_tokens = total_segments = uncovered_segments = 0

    for _, row in use.iterrows():
        segments = split_context_sentences(
            row["context"], row["sentence"] if include_current else None
        )
        for segment in segments:
            tokens = coverage_tokens(segment)
            if not tokens:
                continue
            total_segments += 1
            total_tokens += len(tokens)
            segment_covered = sum(token in VAD_LEX for token in tokens)
            covered_tokens += segment_covered
            uncovered_segments += int(segment_covered == 0)

    return {
        "task": task_name,
        "split": split_name,
        "instances": len(use),
        "segments": total_segments,
        "tokens": total_tokens,
        "covered_tokens": covered_tokens,
        "token_coverage_percent": 100 * covered_tokens / total_tokens,
        "zero_coverage_segments": uncovered_segments,
        "zero_coverage_segment_percent": 100 * uncovered_segments / total_segments,
    }


coverage_rows = []
for task_name in ["state_current", "forecast_next", "forecast_change_subset"]:
    coverage_rows.append(calculate_vad_coverage(dev, task_name, "development"))
    coverage_rows.append(calculate_vad_coverage(test, task_name, "test"))

nrc_vad_coverage = pd.DataFrame(coverage_rows)
coverage_path = os.path.join(RESULTS_DIR, "nrc_vad_coverage_by_task.csv")
nrc_vad_coverage.to_csv(coverage_path, index=False)
display(nrc_vad_coverage.round(2))
print("Saved:", coverage_path)
